[English](../english/02_mcp/02_mcp-practice.ipynb) | 简体中文

本文件是中文母本，英文版见首行链接。启动方式、端口与主脚本以各服务 README 为准；课件截图为原课程历史素材，图中旧路径与中文标签仅供历史参考。

# MCP

## MCP技术本质：Function Calling的更高层实现

传统的Function Calling技术要求围绕不同的外部工具API单独创建一个外部函数，类似一把锁单独配一把钥匙，而一个智能体又往往涉及到多个外部工具设计，因此开发工作量很大。

<img src="../../../assets/FC06.png" alt="FC03" style="width:50%;" />

而MCP技术，全称为Model Context Protocol，模型上下文协议，是一种开发者共同遵守的协议，在这个协议框架下，大家围绕某个API开发的外部工具就能够共用，从而大幅减少重复造轮子的时间。

<img src="../../../assets/FC07.png" alt="FC03" style="width:50%;" />

## MCP调用演示

### 官网的demo

https://github.com/modelcontextprotocol/python-sdk

01_mcp-server-demo/add-mcp-server.py

```python
# 官网的demo：https://github.com/modelcontextprotocol/python-sdk
"""
FastMCP quickstart example.

Run from the repository root:
    uv run examples/snippets/servers/fastmcp_quickstart.py
"""

from mcp.server.fastmcp import FastMCP

# Create an MCP server
mcp = FastMCP("Demo", json_response=True, host="0.0.0.0", port=8000)


# Add an addition tool
@mcp.tool()
def add(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b


# Add a dynamic greeting resource
@mcp.resource("greeting://{name}")
def get_greeting(name: str) -> str:
    """Get a personalized greeting"""
    return f"Hello, {name}!"


# Add a prompt
@mcp.prompt()
def greet_user(name: str, style: str = "friendly") -> str:
    """Generate a greeting prompt"""
    styles = {
        "friendly": "Please write a warm, friendly greeting",
        "formal": "Please write a formal, professional greeting",
        "casual": "Please write a casual, relaxed greeting",
    }

    return f"{styles.get(style, styles['friendly'])} for someone named {name}."


# Run with streamable HTTP transport
if __name__ == "__main__":
    mcp.run(transport="streamable-http")

```

cline_mcp_settings.json配置文件
```json
{
  "mcpServers": {
    "DemoServer": {
      "timeout": 60,
      "type": "streamableHttp",
      "url": "http://127.0.0.1:8000/mcp",
      "command": "",
      "args": [],
      "env": {}
    }
  }
}

### get_weather MCP 版本
`MCP-Demo/02_weather-mcp-server/weather-mcp-server.py`

```python
# -------------------------------
# 工具函数：查询天气（使用OpenWeather API）
# -------------------------------

import json
import os
import requests
from dotenv import load_dotenv, find_dotenv
from mcp.server.fastmcp import FastMCP

# 加载环境变量
_ = load_dotenv(find_dotenv())

# 创建 MCP 服务器实例
mcp = FastMCP("WeatherMCPServer",port="16681")

@mcp.tool()
def get_weather(location: str):
    """
    查询指定城市实时天气，根据输入的城市名称，查询对应城市的实时天气，返回结果将遵循固定的JSON格式，包含城市、温度、天气状况、湿度、风速和更新时间等字段。
    中国城市名称需用英文，如 Beijing。
    """
    open_weather_key = os.environ["API_KEY_OPENWEATHER"]
    open_weather_url = os.environ["BASE_URL_OPENWEATHER"]

    params = {
        "q": location,
        "appid": open_weather_key,
        "units": "metric",
        "lang": "zh_cn"
    }

    response = requests.get(open_weather_url, params=params)
    data = response.json()

    # 直接返回 JSON（MCP 会自动序列化)
    return data


# -------------------------------
# 启动 MCP Server
# -------------------------------
if __name__ == "__main__":
    # 支持 streamable-http（Claude Desktop、浏览器触发）模式
    mcp.run(transport="sse")
```

cline_mcp_settings.json配置文件
```json
{
    "mcpServers": {
        "WeatherMCPServer": {
            "name": "WeatherMCPServer",
            "description": "支持中文城市名的天气查询",
            "url": "http://120.79.252.32:16681/mcp",
            "command": "",
            "args": [],
            "env": {},
            "isActive": true,
            "type": "sse"
        }
    }
}
```

## MCP服务端的三种部署模式

| 特性/模式 | STDIO (本地)             | HTTP/Streamable HTTP (远程)               | SSE (远程，渐趋淘汰)         |
| --------- | ------------------------ | ----------------------------------------- | ---------------------------- |
| 通信方式  | 进程间标准输入输出       | HTTP协议，支持双向流式通信                | HTTP协议，服务端单向推送事件 |
| 部署位置  | 与客户端在同一台机器     | 独立的远程服务器 (如云服务器、Serverless) | 独立的远程服务器             |
| 关键优势  | 简单，零网络延迟         | 网络可访问、多客户端并发、云原生友好      | 实现相对简单                 |
| 主要局限  | 无法远程访问，无网络隔离 | 需要网络配置，有网络延迟                  | 长连接资源消耗高，稳定性较差 |
| 适用场景  | 本地开发、桌面应用集成   | 生产环境、微服务架构                      | 对早期SSE协议兼容            |

### Stdio模式

✨ 特点
- 双向通信（客户端像运行本地进程一样与 MCP Server 通信）
- 通过 stdin / stdout 传递 JSON-RPC
- 性能最好
- 无需网络
- VSCode、Claude Desktop 默认使用它

🧩 场景

- 本地开发工具
- 命令行工具
- 当 MCP Server 以本地可执行文件形式提供时

TimeServer_stdio.py

```python
import json
import time
from datetime import datetime
from mcp.server.fastmcp import FastMCP

# 创建 MCP 服务器实例
mcp = FastMCP("TimeServer")


# 使用装饰器定义获取当前时间的工具函数
@mcp.tool()
def get_current_time():
    current_time = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    return {"current_time": current_time}


# 运行 MCP 服务器,stdio方式运行
if __name__ == "__main__":
    # stdio方式运行
    mcp.run(transport="stdio")
```

cline 注册方式（STDIO）

```json
{
    "mcpServers": {
        "TimeServer": {
            "command": "/opt/miniconda3/envs/env_agent/bin/python",
            "args": [
                "/workspace/learn-llm-agent-easily/03_AI原生的Agent实操/03_file-agent-mcp/TimeAgentMCP.py"
            ],
            "autoApprove": [
                "get_current_time"
            ],
            "disabled": false
        }
    }
}

```


### SSE（Server-Sent Events）事件推送

✨ 特点

- 单向通信：Server → Client
- 基于 HTTP 长连接
- 类似订阅某些事件
- 不用于普通 RPC 调用，只用于 事件流

🧩 场景
- 模型后台任务状态更新
- 进度条 push
- 通知/异步事件

TimeServer_sse.py

```python
import json
import time
from datetime import datetime
from mcp.server.fastmcp import FastMCP

# 创建 MCP 服务器实例
mcp = FastMCP("TimeServer")


# 使用装饰器定义获取当前时间的工具函数
@mcp.tool()
def get_current_time():
    current_time = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    return {"current_time": current_time}


if __name__ == "__main__":
    # sse方式运行
    mcp.run(transport="sse")
```

cline 注册方式（SSE）

```json
{
    "mcpServers": {
        "TimeServer": {
            "name": "TimeServer",
            "description": "",
            "baseUrl": "http://127.0.0.1:8000/sse",
            "command": "",
            "args": [],
            "env": {},
            "isActive": true,
            "type": "sse"
        }
    }
}
```

### Streamable HTTP（远程可访问的 MCP Server）（推荐方式）

✨ 特点

- 支持 chunked transfer（流式输出）
- 类似 OpenAI ChatCompletions（流式 token）
- 正式文档称 可流式 HTTP

🧩 场景
- 远程大模型
- 流式工具返回（例如：逐行返回日志）

TimeServer_http_stream.py

```python
import json
import time
from datetime import datetime
from mcp.server.fastmcp import FastMCP

# 创建 MCP 服务器实例
mcp = FastMCP("TimeServer")


# 使用装饰器定义获取当前时间的工具函数
@mcp.tool()
def get_current_time():
    current_time = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    return {"current_time": current_time}

if __name__ == "__main__":
    # streamable-http方式运行
    mcp.run(transport="streamable-http")
```

cline 注册方式（Streamable HTTP）

```json
{
    "mcpServers": {
        "TimeServer": {
            "name": "TimeServer",
            "description": "",
            "url": "http://127.0.0.1:8000/mcp",
            "command": "",
            "args": [],
            "env": {},
            "isActive": true,
            "type": "streamableHttp"
        }
    }
}
```

## 本地文件操作MCP服务搭建

#### MCP服务端搭建
文件夹中包括： 

- 脚本（.py）
- 被操作的数据（dataset）

<img src="../../../assets/filemcp.png" alt="FC03" style="width:20%;" />

file_agent_mcp_server.py，这里启用streamable-http方式，服务端口指定为8001

```python
...
...
# 1️⃣ 创建 MCP 服务器实例
mcp = FastMCP("FileAgentServer", port=8001)

...
...

# 5️⃣ 启动 MCP 服务器
if __name__ == "__main__":
    mcp.run(transport="streamable-http")

```

启动服务，日志信息如下： 

```bash
conda activate env_agent
(base) root@DESKTOP-WILL:/workspace/learn-llm-agent-easily# conda activate env_agent
(env_agent) root@DESKTOP-WILL:/workspace/learn-llm-agent-easily# /opt/miniconda3/envs/env_agent/bin/python /workspace/learn-llm-agent-easily/03_AI原生的Agent实操/03_file-agent-mcp/file_agent_mcp_server.py
INFO:     Started server process [259578]
INFO:     Waiting for application startup.
[12/02/25 10:55:37] INFO     StreamableHTTP session manager started                                                                                                streamable_http_manager.py:110
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8001 (Press CTRL+C to quit)


```


<img src="../../../assets/filemcp2.png" alt="FC03" style="width:80%;" />

### MCP客户端（Cline）

除了手搓客户端外，市面上有很多商用的成熟客户端，这里以Cline为例

#### 安装
在VSCode - Extensions 中搜索 Cline 完成安装

#### 配置MCPServer
打开Cline，点击MCP Server，点击Configure MCP Servers， 弹出 cline_mcp_settings.json 文件，新增MCPServer

```json
{
    "mcpServers": {
        "FileAgentServer": {
            "name": "FileAgentServer",
            "description": "",
            "url": "http://127.0.0.1:8001/mcp",
            "command": "",
            "args": [],
            "env": {},
            "isActive": true,
            "type": "streamableHttp"
        }
    }
}
```

<img src="../../../assets/filemcp3.png" alt="FC03" style="width:100%;" />

添加MCP Server成功

<img src="../../../assets/filemcp4.png" alt="FC03" style="width:40%;" />

#### 配置大模型

这里cline了免费的模型，选择Cline

<img src="../../../assets/filemcp5.png" alt="FC03" style="width:40%;" />

如果需要配置本地部署的模型，或者硅基流动提供的模型，选择OpenAI Compatiable

<img src="../../../assets/filemcp6.png" alt="FC03" style="width:40%;" />

#### 实操演示

给出提示词： 

```bash
/workspace/learn-llm-agent-easily/03_AI原生的Agent实操/03_file-agent-mcp/dataset ， 列出这个目录下所有文件，读取文件内容，根据所用编程语言给文件取合适的名字
```

<img src="../../../assets/filemcp7.png" alt="FC03" style="width:100%;" />

<img src="../../../assets/filemcp8.png" alt="FC03" style="width:100%;" />

给出提示词： 

```bash
恢复成文件原来的样子
```

<img src="../../../assets/filemcp9.png" alt="FC03" style="width:100%;" />

### MCP Server在后台运行

✅ **使用 nohup**

适合长期运行 + 自动写日志。

```bash
nohup /opt/miniconda3/envs/env_agent/bin/python \
  /workspace/learn-llm-agent-easily/03_AI原生的Agent实操/03_file-agent-mcp/file_agent_mcp_server.py \
  > mcp_server.log 2>&1 &
```

查看是否在运行：

```bash
ps -ef | grep file_agent_mcp_server
```

停止：

```bash
kill -9 <PID>
```

实时查看日志
```bash
tail -f mcp_server.log
```

## MCP实战

### weather-mcp
[天气查询MCP](cases/01_weather-mcp-server/)

### file-mcp

[文件操作wMCP](cases/02_file-mcp-server/)

## 本机验证记录（2026-10-18，`env_agent_uv` 环境）

离线回归按文档工作目录实跑，天气与双语 26 项、文件服务 21 项、Skill 对比 4 项，共 51 项全部通过。测试对
OpenWeather 打桩并使用临时文件，不消耗真实 API 额度。

**预期输出：**

```text
# 02_mcp/cases
Ran 26 tests in 5.027s
OK
Ran 21 tests in 2.769s
OK

# 03_skills/cases/01_weather_skill
Ran 4 tests in 1.491s
OK
```

四个 MCP 服务在本机实际启动并完成调用，下列工具清单与返回值均为真实结果；`w-OpenWeatherMCP` 在未配置密钥时
如实返回 `missing_api_key`，与 Skill 直连脚本的诊断一致。

**预期输出：**

```text
w-MCPDemo          streamable-http  tools: ping, add                       prompts: greet_user
w-TimeMCP          stdio            tools: get_current_time, convert_time
w-OpenWeatherMCP   streamable-http  tools: ping, check_config, get_weather,
                                      get_weather_by_coordinates, get_forecast,
                                      get_forecast_by_coordinates
w-FileWorkbenchMCP streamable-http  tools: 18 个（ping、list_allowed_roots、list_files、read_file、
                                      rename_file、删除恢复与目录快照等）

$ add(a=29, b=13)
42
$ read_resource("greeting://张三/zh")
你好，%E5%BC%A0%E4%B8%89！
$ get_current_time(timezone="Asia/Shanghai", language="zh")
{"ok": true, "current_time": "2026-10-18 22:36:35", "timezone": "CST", "utc_offset_seconds": 28800}
$ convert_time(iso_time="2026-10-18T18:30:00", from_timezone="Asia/Shanghai", to_timezone="America/New_York")
{"ok": true, "target_iso8601": "2026-10-18T06:30:00-04:00"}
$ list_files(base_dir="02_file-mcp-server/files", language="zh", limit=3)
{"ok": true, "count": 3, "total": 5}
$ check_config(language="zh")                      # 未配置 OPENWEATHER_API_KEY 时
{"ok": false, "error": "missing_api_key"}
```
